# Getting Started with iPSC Digital Twin

This notebook provides an introduction to the hybrid physics-ML digital twin framework for iPSC differentiation.

## Contents
1. ODE-based mechanistic simulation
2. Data generation for ML training
3. Training LSTM predictors
4. Digital twin with ML integration
5. Prediction and uncertainty quantification

In [ ]:
# Setup
import sys
from pathlib import Path

# Add project root to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from src.models.simulators import iPSCDifferentiationSimulator, StochasticiPSCSimulator
from src.models.digital_twin import DigitalTwinEngine
from src.models.predictors import LSTMPredictor
from src.data.data_generator import SyntheticDataGenerator
from src.visualization import DigitalTwinPlotter
from src.utils import load_config

%matplotlib inline

# Load configuration
config = load_config()
plotter = DigitalTwinPlotter()

## 1. Mechanistic ODE Simulator

The core of our digital twin is a mechanistic model based on ordinary differential equations (ODEs) that captures:
- Pluripotency marker dynamics (OCT4, NANOG, SOX2)
- Differentiation marker expression
- Cell population growth
- Response to growth factors (FGF2, Retinoic Acid)

In [ ]:
# Create simulator
simulator = iPSCDifferentiationSimulator(config)

# Scenario 1: Maintain pluripotency (high FGF2)
time1, states1 = simulator.run_simulation(
    duration=14,
    timesteps=100,
    growth_factors={'fgf2': 1.0, 'retinoic_acid': 0.0}
)

simulator.reset()

# Scenario 2: Induce differentiation (high RA)
time2, states2 = simulator.run_simulation(
    duration=14,
    timesteps=100,
    growth_factors={'fgf2': 0.0, 'retinoic_acid': 1.0}
)

# Plot comparison
plotter.plot_comparison(
    time1,
    [states1, states2],
    labels=['High FGF2 (Pluripotency)', 'High RA (Differentiation)'],
    title='ODE Simulation: Different Protocols'
)

## 2. Generate Training Data

We use the ODE simulator to generate diverse training data for ML models by varying:
- Initial conditions
- Growth factor protocols
- Adding measurement noise

In [ ]:
# Create data generator
simulator.reset()
data_generator = SyntheticDataGenerator(simulator, config)

# Generate a single random trajectory
time_points, states = data_generator.generate_trajectory(
    timesteps=100,
    add_noise=True,
    noise_level=0.02
)

# Visualize
plotter.plot_trajectory(
    time_points,
    states,
    title='Example Training Trajectory with Noise'
)

print(f"Final state:")
print(f"  Pluripotency: {states[-1, 0]:.3f}")
print(f"  Differentiation: {states[-1, 1]:.3f}")
print(f"  Population: {states[-1, 2]:.0f}")

## 3. Digital Twin with Real-Time Updates

The digital twin maintains a virtual replica that:
- Tracks current state
- Updates in real-time
- Predicts future outcomes
- Provides recommendations

In [ ]:
# Create digital twin
simulator.reset()
twin = DigitalTwinEngine(simulator, config=config)

# Initialize
twin.initialize(growth_factors={'fgf2': 1.0, 'retinoic_acid': 0.0})
print(f"Initialized at t=0: {twin.current_state}")

# Simulate real-time updates (7 days)
print("\nSimulating 7 days of culture...")
for day in range(1, 8):
    state = twin.update(
        duration=24,  # 24 hours
        growth_factors={'fgf2': 1.0, 'retinoic_acid': 0.0}
    )
    print(f"Day {day}: P={state['pluripotency']:.3f}, "
          f"D={state['differentiation']:.3f}, N={state['population']:.0f}")
    
    # Get recommendations
    if day % 2 == 0:
        recs = twin.recommend_action()
        print(f"  → {recs['actions'][0]['message']}")

# Predict next 48 hours
print("\nPredicting next 48 hours with RA...")
prediction = twin.predict(
    horizon=48,
    growth_factors={'fgf2': 0.0, 'retinoic_acid': 1.0}
)

final_pred = prediction['predicted_states'][-1]
print(f"Predicted (48h): P={final_pred['pluripotency']:.3f}, "
      f"D={final_pred['differentiation']:.3f}")

## 4. Stochastic Simulation

For uncertainty quantification, we use stochastic simulations that model:
- Gene expression noise
- Cell-to-cell variability
- Measurement uncertainty

In [ ]:
# Create stochastic simulator
stochastic_sim = StochasticiPSCSimulator(config)

# Run ensemble of realizations
time_points, ensemble_states = stochastic_sim.run_simulation(
    duration=14,
    timesteps=100,
    growth_factors={'fgf2': 0.0, 'retinoic_acid': 1.0},
    add_noise=True,
    n_realizations=20
)

# Compute statistics
stats = stochastic_sim.get_ensemble_statistics(ensemble_states)

# Plot with uncertainty
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pluripotency
axes[0].plot(time_points, stats['mean'][:, 0], 'b-', linewidth=2, label='Mean')
axes[0].fill_between(time_points, stats['q05'][:, 0], stats['q95'][:, 0],
                      alpha=0.3, color='blue', label='90% CI')
axes[0].set_xlabel('Time (days)')
axes[0].set_ylabel('Pluripotency')
axes[0].set_title('Pluripotency with Uncertainty')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Differentiation
axes[1].plot(time_points, stats['mean'][:, 1], 'r-', linewidth=2, label='Mean')
axes[1].fill_between(time_points, stats['q05'][:, 1], stats['q95'][:, 1],
                      alpha=0.3, color='red', label='90% CI')
axes[1].set_xlabel('Time (days)')
axes[1].set_ylabel('Differentiation')
axes[1].set_title('Differentiation with Uncertainty')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Next Steps

To complete the hybrid ML-physics system:

1. **Train ML Predictor**: Run `experiments/train_predictor.py` to train an LSTM model
2. **Load ML into Twin**: Use `twin.load_ml_predictor()` to integrate trained model
3. **Compare Methods**: Use `twin.compare_prediction_methods()` to evaluate hybrid approach
4. **Optimize Protocols**: Use predictions to find optimal differentiation protocols

See `examples/hybrid_ml_demo.py` for a complete demonstration.

## Summary

This notebook demonstrated:
- ✓ ODE-based mechanistic simulation
- ✓ Synthetic data generation
- ✓ Digital twin with real-time updates
- ✓ Stochastic simulation and uncertainty quantification

The framework is ready for:
- ML model training
- Hybrid physics-ML predictions
- Protocol optimization
- Integration with real experimental data